In [ ]:
from mg_sampling import PROJECT_DIR
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from ast import literal_eval
import itertools

### Inner-bound (< 2.8 Å)

In [ ]:
# Replex
systems = ['7ez0', '7ez0_noion']
forcefields = ['mMg_tip3p']
replicates = [0]
repids = [0]
ions = ['MG', 'NA']

df = []
for system, forcefield, replicate, repid, ion in itertools.product(systems, forcefields, replicates, repids, ions):
    path = PROJECT_DIR / 'replex' / system / forcefield / f'replicate{replicate}' / f'rep{repid:02d}'
    _df = pd.read_csv(path / f'{ion}_motifs.csv', na_filter=False)
    _df = _df.loc[_df['time'] % 1000.0 == 0]
    _df['system'] = system
    _df['forcefield'] = forcefield
    _df['repid'] = repid
    _df['replicate'] = replicate
    _df['method'] = 'replex'
    df.append(_df)

# PlainMD
systems = ['7ez0', '7ez0_noion']
forcefields = ['nMg_tip3p_hard', 'mMg_tip3p_hard', 'nMg_tip3p']
replicates = range(8)
ions = ['MG', 'NA']

for system, forcefield, replicate, repid, ion in itertools.product(systems, forcefields, replicates, repids, ions):
    try:
        path = PROJECT_DIR / 'plain_md' / system / forcefield / f'rep{replicate:02d}'
        _df = pd.read_csv(path / f'{ion}_motifs.csv', na_filter=False)
        _df = _df.loc[_df['time'] % 1000.0 == 0]
        _df['system'] = system
        _df['forcefield'] = forcefield
        _df['replicate'] = replicate
        _df['method'] = 'plain_md'
        df.append(_df)
    except FileNotFoundError as e:
        print(e)
        pass

df = pd.concat(df)
for col in ['name', 'mgrna', 'forcefield', 'system', 'method']:
    df[col] = df[col].astype('category')


In [ ]:
tstart_replex = 50000  # ps
tstart_plain_md = 150000

grouped = df.query(f"(method == 'replex' and time >= {tstart_replex}) or (method == 'plain_md' and time >= {tstart_plain_md})").groupby(
    ['time', 'name', 'n_inner', 'forcefield', 'system', 'replicate', 'method'], observed=True
    ).size().reset_index(name='counts')

In [ ]:
# Compute unbound by taking the complement
n_tot = {'MG': 178, 'NA': 121}
tot = grouped.groupby(['time', 'name', 'forcefield', 'system', 'replicate', 'method'], observed=True).sum()['counts'].reset_index()
tot['tot'] = tot['name'].map(n_tot).astype(int)
tot['counts'] = tot['tot'] - tot['counts']
tot = tot.drop(columns='tot')
tot.insert(2, 'n_inner', -1)
grouped = pd.concat([grouped, tot])
grouped['sim'] = grouped['method'].str.cat(grouped['forcefield'], sep='_')
grouped['sim'] = grouped['sim'].map({
    'replex_mMg_tip3p': 'μMg (REx)',
    'plain_md_mMg_tip3p_hard': 'μMg (MD)',
    'plain_md_nMg_tip3p_hard': 'nMg (MD)',
    'plain_md_nMg_tip3p': 'nMg (MD-soft)',
})
grouped['binding'] = grouped['n_inner'].replace(-1, 'Bulk').replace(0, 'Outer')
grouped = grouped.sort_values('n_inner')

In [ ]:
grouped.query("forcefield == 'nMg_tip3p'")

In [ ]:
palette = ['tomato', 'gray', 'mediumpurple', 'mediumpurple']
labels = ['μMg (REx)', 'μMg (MD)', 'nMg (MD)', 'nMg (MD-soft)']

g = sns.FacetGrid(grouped, row='name', col='system', sharey='row')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    y='binding',
    x='counts',
    orient='h',
    hue='sim',
    hue_order=labels,
    palette=palette,
)

g.add_legend(g._legend_data, loc='center', bbox_to_anchor=(0.5, 1), ncol=len(labels))
g.set_xlabels("Avg. number")
g.set_ylabels(r"Coordination")
g.set_titles("{row_name} ({col_name})")
plt.gcf().set_size_inches((5, 5))
plt.tight_layout()
plt.savefig('figures/average_coordination.pdf', bbox_inches='tight')

## Inner-bound sites

In [ ]:
tstart_replex = 50000  # ps
tstart_plain_md = 150000

grouped = df.query(f"((method == 'replex' and time >= {tstart_replex}) or (method == 'plain_md' and time >= {tstart_plain_md})) and n_inner > 0").copy()
grouped['inner_names'] = grouped['inner_names'].map(literal_eval)

In [ ]:
grouped = grouped.explode('inner_names')

In [ ]:
# Count by name
grouped = grouped.groupby(['name', 'system', 'method', 'forcefield', 'inner_names'], observed=True).size()


In [ ]:
grouped.groupby(['name', 'system', 'method', 'forcefield'], observed=True).sum()

In [ ]:
grouped = (grouped / grouped.groupby(['name', 'system', 'method', 'forcefield'], observed=True).sum())

In [ ]:
grouped = grouped.reset_index(name='proportion')

In [ ]:
grouped

In [ ]:
grouped['sim'] = grouped['method'].str.cat(grouped['forcefield'], sep='_')
grouped['sim'] = grouped['sim'].map({
    'replex_mMg_tip3p': 'μMg (REx)',
    'plain_md_mMg_tip3p_hard': 'μMg (MD)',
    'plain_md_nMg_tip3p_hard': 'nMg (MD)',
})
palette = ['tomato', 'gray', 'mediumpurple']
labels = ['μMg (REx)', 'μMg (MD)', 'nMg (MD)']

g = sns.FacetGrid(grouped, row='name', col='system', sharey='row')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    y='inner_names',
    x='proportion',
    orient='h',
    hue='sim',
    hue_order=labels,
    palette=palette,
    order=grouped[grouped['proportion'] > 0.01].sort_values('proportion', ascending=False)['inner_names'].unique(),
)

g.add_legend(g._legend_data, loc='center', bbox_to_anchor=(0.5, 1), ncol=len(labels))
g.set_xlabels("Proportion")
g.set_ylabels("Inner-bound atom type")
g.set_titles("{row_name} ({col_name})")
for axis in g.axes.flat:
    axis.tick_params(labelleft=True)
plt.gcf().set_size_inches((5, 5))
plt.tight_layout()
plt.savefig('figures/binding_sites.pdf', bbox_inches='tight')